# TE-PAI vs. Trotter — snapshot estimation with error bars

We evolve a **6-qubit Heisenberg spin chain** from a Néel state and estimate
$\langle Z_0(t)\rangle$:

1. **Exact** — statevector expectation of the first-order Trotter circuit (reference line).
2. **Trotter** — the (deep) Trotter circuit measured with `Ns` shots.
3. **TE-PAI** — `M` shallow random circuits, signed-weighted; evaluated **in parallel across all CPU cores** via `TEPAI.estimate`.

Both are unbiased for the Trotter-evolved value, so they track the exact curve within error bars. TE-PAI's error bars come from the quasiprobability **overhead** $\gamma$ (the price of shallower circuits), not from shot noise.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from pai_shadow.hamil import Heisenberg_Hamil
from pai_shadow.backend import get_backend
from pai_shadow.trotter import trotter_circuit
from pai_shadow.te_pai import TEPAI

np.random.seed(0)

In [ ]:
# 6-qubit Heisenberg spin chain
n_qubits = 6
H = Heisenberg_Hamil(n_qubits, 1.0, 1.0, 1.0)
backend = get_backend("qulacs")

# Neel initial state |0101010>
neel = [q % 2 for q in range(n_qubits)]
idx = sum(b << q for q, b in enumerate(neel))
psi0 = np.zeros(1 << n_qubits, dtype=complex)
psi0[idx] = 1.0

observable = "Z" + "I" * (n_qubits - 1)   # <Z_0>

# parameters (n_steps large enough that 2|coef|*dt <= delta for TE-PAI)
n_steps = 40
delta   = np.pi / 32
M       = 8000   # TE-PAI circuits (evaluated in parallel)
Ns      = 3000   # Trotter measurement shots
times   = np.linspace(0.0, 1.5, 11)

def z0(bitstring):
    return 1 - 2 * int(bitstring[-1])  # Z eigenvalue on qubit 0 (right-most bit)

In [ ]:
exact, trot_mean, trot_err, tepai_mean, tepai_err = [], [], [], [], []
for t in times:
    circ = trotter_circuit(H, t, n_steps, init_state=psi0)
    exact.append(backend.expectation(circ, observable))
    # Trotter with finite measurement shots
    v = np.array([z0(b) for b in backend.sample(circ, Ns)])
    trot_mean.append(v.mean()); trot_err.append(v.std() / np.sqrt(Ns))
    # TE-PAI: M random circuits, evaluated in parallel across all cores
    tp = TEPAI(H, delta, t, n_steps, init_state=psi0)
    w = tp.estimate(observable, M, backend="qulacs", n_jobs=os.cpu_count())
    tepai_mean.append(w.mean()); tepai_err.append(w.std() / np.sqrt(M))
    print(f"t={t:.2f}  exact={exact[-1]:+.3f}  overhead={tp.overhead:.2f}")

In [ ]:
plt.figure(figsize=(9, 5))
k = 2  # plot 95% confidence (+/- 2 sigma)
plt.plot(times, exact, "k-", lw=2, label="exact (statevector)")
plt.errorbar(times, trot_mean, yerr=k*np.array(trot_err), fmt="o", capsize=3,
             color="tab:blue", label=f"Trotter, {Ns} shots")
plt.errorbar(times, tepai_mean, yerr=k*np.array(tepai_err), fmt="s", capsize=3,
             color="tab:red", label=f"TE-PAI, M={M}")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("time $t$"); plt.ylabel(r"$\langle Z_0(t)\rangle$")
plt.title(r"7-qubit Heisenberg: Trotter vs TE-PAI ($\pm2\sigma$ error bars)")
plt.legend(); plt.tight_layout(); plt.show()

## Notes

- `TEPAI.estimate(...)` fuses circuit **generation + evaluation inside worker processes** and splits the work over `n_jobs` cores (a persistent pool is reused across the time points, so process startup is paid only once).
- It returns the per-circuit weighted values; the **mean** is the unbiased estimate and **std/√M** the error bar.
- Pass `shots=<int>` to `estimate` for literal single-shot measurement snapshots (Z/I observables); the default `shots=None` uses each circuit's exact expectation (lower variance, faster convergence).
- Lower `delta` or raise `n_steps` to see how `overhead` and the TE-PAI error bars change.

- Error bars show **95% confidence ($\pm2\sigma$)**. With $\pm1\sigma$ bars the exact value lies inside only ~68% of the points *by definition* — that is expected, not a bug.

## The same simulation, with gate noise

Repeat the same $\langle Z_0(t)\rangle$ sweep with depolarizing gate noise, treated **exactly**
by **density-matrix** simulation. The deep Trotter circuit is pulled toward 0, while TE-PAI keeps
tracking the exact line — the noise-robustness of TE-PAI's shallow circuits.

TE-PAI still samples its random circuits (its quasiprobability variance is intrinsic), but each
circuit's noisy value is evaluated **exactly** (no shot noise). A density matrix is $2^n$ times
larger than a state vector, so we use a modest system (6 qubits) and `M`. Change `noise.kind` to
explore `bitflip` / `phaseflip` / `amplitude_damping`.

In [ ]:
from pai_shadow.backend import NoiseSpec

noise       = NoiseSpec(p1=5e-4, p2=5e-3, kind="depolarizing")
noisy       = get_backend("qulacs", noise=noise)
M_noise     = 500     # density-matrix noise costs ~2**n per circuit
times_noise = np.linspace(0.0, 0.9, 7)   # focus where <Z0> is large (shallow TE-PAI circuits)

exactN, trotN, tepaiN, tepaiN_err = [], [], [], []
for t in times_noise:
    circ = trotter_circuit(H, t, n_steps, init_state=psi0)
    exactN.append(backend.expectation(circ, observable))          # noiseless reference
    trotN.append(noisy.expectation(circ, observable))             # Trotter: exact noisy
    tp = TEPAI(H, delta, t, n_steps, init_state=psi0)
    w = tp.estimate(observable, M_noise, n_jobs=os.cpu_count(), noise=noise)  # each circuit exact (density)
    tepaiN.append(w.mean()); tepaiN_err.append(w.std() / np.sqrt(M_noise))
    print(f"t={t:.2f}  exact={exactN[-1]:+.3f}  Trotter={trotN[-1]:+.3f}  TE-PAI={tepaiN[-1]:+.3f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(times_noise, exactN, "k-", lw=2, label="exact (noiseless)")
plt.plot(times_noise, trotN, "o-", color="tab:blue", label="Trotter (noisy, density-matrix)")
plt.errorbar(times_noise, tepaiN, yerr=2*np.array(tepaiN_err), fmt="s-", capsize=3,
             color="tab:red", label="TE-PAI (noisy, density-matrix)")
plt.xlabel("time $t$"); plt.ylabel(r"$\langle Z_0(t)\rangle$")
plt.title(f"Exact density-matrix {noise.kind} noise ($p_2={noise.p2}$)")
plt.legend(); plt.tight_layout(); plt.show()

### Notes

- Both curves use **exact density-matrix** noise (no shot noise): Trotter is one density evolution; each TE-PAI circuit is evaluated exactly.
- The only statistical bars are TE-PAI's **intrinsic quasiprobability variance** (`std/sqrt(M)`), not measurement shot noise.
- Density-matrix cost grows as $2^n$ per circuit, so raising the qubit count or `M` gets expensive quickly; lower them if it is slow.